Basic dataset overview
- Total number of records
- Exact duplicate pairs
- Percentage of unchanged pairs, where text == standardized

In [18]:
import json
import pandas as pd


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def total_records(df):
    """Return the total number of records."""
    return len(df)


def exact_duplicate_pairs(df):
    """Count duplicate text-standardized pairs, excluding the first occurrence."""
    return df.duplicated(
        subset=["text", "standardized"],
        keep="first"
    ).sum()


def duplicate_source_texts(df):
    """Count duplicate source texts, excluding the first occurrence."""
    return df.duplicated(
        subset=["text"],
        keep="first"
    ).sum()


def duplicate_target_texts(df):
    """Count duplicate target texts, excluding the first occurrence."""
    return df.duplicated(
        subset=["standardized"],
        keep="first"
    ).sum()


def unchanged_pairs_percentage(df):
    """Calculate the percentage of pairs where text equals standardized."""
    if len(df) == 0:
        return 0.0

    unchanged_count = (df["text"] == df["standardized"]).sum()

    return (unchanged_count / len(df)) * 100


def main():
    df = load_dataset(FILE_PATH)

    print("BASIC DATASET OVERVIEW")
    print("-" * 40)

    print(f"Total number of records: {total_records(df)}")
    print(f"Exact duplicate pairs: {exact_duplicate_pairs(df)}")
    print(f"Duplicate source texts: {duplicate_source_texts(df)}")
    print(f"Duplicate target texts: {duplicate_target_texts(df)}")
    print(
        f"Percentage of unchanged pairs: "
        f"{unchanged_pairs_percentage(df):.2f}%"
    )


if __name__ == "__main__":
    main()

BASIC DATASET OVERVIEW
----------------------------------------
Total number of records: 13830
Exact duplicate pairs: 0
Duplicate source texts: 0
Duplicate target texts: 1938
Percentage of unchanged pairs: 0.95%


Quality assurance

In [19]:
import json
import pandas as pd


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def total_records(df):
    """Return the total number of records."""
    return len(df)


def exact_duplicate_pairs(df, examples=5):
    """Count and display exact duplicate source-target pairs."""
    duplicate_mask = df.duplicated(
        subset=["text", "standardized"],
        keep=False
    )

    duplicate_rows = df.loc[
        duplicate_mask,
        ["id", "text", "standardized"]
    ].sort_values(["text", "standardized"])

    # Count repeated rows after the first occurrence
    duplicate_count = df.duplicated(
        subset=["text", "standardized"],
        keep="first"
    ).sum()

    print(f"\nExact duplicate pairs: {duplicate_count}")
    print("Examples:")
    print(duplicate_rows.head(examples).to_string(index=False))

    return duplicate_count


def duplicate_source_texts(df, examples=5):
    """Count and display source texts appearing more than once."""
    duplicate_mask = df.duplicated(
        subset=["text"],
        keep=False
    )

    duplicate_rows = df.loc[
        duplicate_mask,
        ["id", "text", "standardized"]
    ].sort_values("text")

    duplicate_count = df.duplicated(
        subset=["text"],
        keep="first"
    ).sum()

    print(f"\nDuplicate source texts: {duplicate_count}")
    print("Examples:")
    print(duplicate_rows.head(examples).to_string(index=False))

    return duplicate_count


def duplicate_target_texts(df, examples=5):
    """Count and display target texts appearing more than once."""
    duplicate_mask = df.duplicated(
        subset=["standardized"],
        keep=False
    )

    duplicate_rows = df.loc[
        duplicate_mask,
        ["id", "text", "standardized"]
    ].sort_values("standardized")

    duplicate_count = df.duplicated(
        subset=["standardized"],
        keep="first"
    ).sum()

    print(f"\nDuplicate target texts: {duplicate_count}")
    print("Examples:")
    print(duplicate_rows.head(examples).to_string(index=False))

    return duplicate_count


def unchanged_pairs(df, examples=5):
    """Calculate and display pairs where source equals target."""
    unchanged_rows = df.loc[
        df["text"] == df["standardized"],
        ["id", "text", "standardized"]
    ]

    unchanged_count = len(unchanged_rows)

    if len(df) == 0:
        unchanged_percentage = 0.0
    else:
        unchanged_percentage = (unchanged_count / len(df)) * 100

    print(
        f"\nUnchanged pairs: {unchanged_count} "
        f"({unchanged_percentage:.2f}%)"
    )
    print("Examples:")
    print(unchanged_rows.head(examples).to_string(index=False))

    return unchanged_count, unchanged_percentage


def main():
    df = load_dataset(FILE_PATH)

    print("BASIC DATASET OVERVIEW")
    print("-" * 50)
    print(f"Total number of records: {total_records(df)}")

    # exact_duplicate_pairs(df, examples=10)
    # duplicate_source_texts(df, examples=10)
    # duplicate_target_texts(df, examples=10)
    unchanged_pairs(df, examples=10)


if __name__ == "__main__":
    main()

BASIC DATASET OVERVIEW
--------------------------------------------------
Total number of records: 13830

Unchanged pairs: 132 (0.95%)
Examples:
  id                                           text                                   standardized
  85                 [NAME], la route bajahkoki!! 🤣                 [NAME], la route bajahkoki!! 🤣
 121                     [NAME], 2026 pour toi..😅🤣😂                     [NAME], 2026 pour toi..😅🤣😂
 334                         Pena bon maintenance..                         Pena bon maintenance..
 340 Abuse of power. Btw.. Le haut gradé pena nom?? Abuse of power. Btw.. Le haut gradé pena nom??
 572                         [NAME], moi innocent 😂                         [NAME], moi innocent 😂
 693                                      Komiko!!!                                      Komiko!!!
 768            [NAME], vegetable seller lerla 🤔???            [NAME], vegetable seller lerla 🤔???
 842   [NAME],.. Oui vraimem... Bel problem lerla..   [NAME],..

Sentence-length statistics
Calculate sentence length separately for text and standardized.
Use both:

Number of characters
Number of words/tokens

Report:

Minimum
Maximum
Mean
Median
Standard deviation
25th percentile
75th percentile
95th percentile

Example thesis table:






























MeasurementSource textStandardized textMean words per sentence——Median words per sentence——Maximum words——Mean characters——

Vocab stats
- Number of tokens occurring only once, known as hapax legomena
- Vocabulary size
Number of tokens occurring only once, known as hapax legomena
Number and percentage of rare tokens
Most frequent words
Average token length
Longest tokens
Lexical diversity

Sentence length analysis

In [20]:
import json
import pandas as pd


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def calculate_character_lengths(df):
    """Calculate character lengths for source and standardized texts."""
    df = df.copy()

    df["source_character_count"] = (
        df["text"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    df["target_character_count"] = (
        df["standardized"]
        .fillna("")
        .astype(str)
        .str.len()
    )

    return df


def calculate_word_lengths(df):
    """Calculate word counts using whitespace-based tokenization."""
    df = df.copy()

    df["source_word_count"] = (
        df["text"]
        .fillna("")
        .astype(str)
        .apply(lambda text: len(text.split()))
    )

    df["target_word_count"] = (
        df["standardized"]
        .fillna("")
        .astype(str)
        .apply(lambda text: len(text.split()))
    )

    return df


def calculate_length_statistics(length_series):
    """Calculate descriptive statistics for a length column."""
    return {
        "Minimum": length_series.min(),
        "Maximum": length_series.max(),
        "Mean": length_series.mean(),
        "Median": length_series.median(),
        "Standard deviation": length_series.std(),
        "25th percentile": length_series.quantile(0.25),
        "75th percentile": length_series.quantile(0.75),
        "95th percentile": length_series.quantile(0.95)
    }


def create_sentence_length_report(df):
    """Create the complete sentence-length statistics report."""
    df = calculate_character_lengths(df)
    df = calculate_word_lengths(df)

    report = pd.DataFrame({
        "Source characters": calculate_length_statistics(
            df["source_character_count"]
        ),
        "Target characters": calculate_length_statistics(
            df["target_character_count"]
        ),
        "Source words": calculate_length_statistics(
            df["source_word_count"]
        ),
        "Target words": calculate_length_statistics(
            df["target_word_count"]
        )
    })

    return report


def main():
    df = load_dataset(FILE_PATH)

    report = create_sentence_length_report(df)

    print("SENTENCE-LENGTH STATISTICS")
    print("-" * 80)
    print(report.round(2).to_string())


if __name__ == "__main__":
    main()

SENTENCE-LENGTH STATISTICS
--------------------------------------------------------------------------------
                    Source characters  Target characters  Source words  Target words
Minimum                          2.00               3.00          1.00          1.00
Maximum                       1261.00            1246.00        231.00        230.00
Mean                            90.67              91.75         17.70         17.51
Median                          62.00              63.00         12.00         12.00
Standard deviation              94.20              94.64         18.11         17.93
25th percentile                 36.00              37.00          7.00          7.00
75th percentile                110.00             111.00         21.00         21.00
95th percentile                261.00             262.00         51.00         50.00


3. Vocabulary statistics

In [29]:
import json
import re
import pandas as pd
from collections import Counter


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"

RARE_TOKEN_THRESHOLD = 2
TOP_N = 20


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def tokenize(text):
    """
    Convert text to lowercase and extract word tokens.

    Punctuation and emojis are excluded.
    Apostrophes inside words are preserved.
    """
    if pd.isna(text):
        return []

    text = str(text).lower()

    return re.findall(
        r"\b\w+(?:['’]\w+)*\b",
        text,
        flags=re.UNICODE
    )


def get_all_tokens(df, column_name):
    """Extract all tokens from one column without using a loop."""
    token_lists = df[column_name].apply(tokenize)

    return [
        token
        for token_list in token_lists
        for token in token_list
    ]


def calculate_vocabulary_statistics(tokens):
    """Calculate vocabulary statistics."""
    frequencies = Counter(tokens)

    total_count = len(tokens)
    unique_count = len(frequencies)

    hapax_count = sum(
        1 for frequency in frequencies.values()
        if frequency == 1
    )

    rare_count = sum(
        1 for frequency in frequencies.values()
        if frequency <= RARE_TOKEN_THRESHOLD
    )

    if unique_count > 0:
        rare_percentage = (rare_count / unique_count) * 100
    else:
        rare_percentage = 0.0

    if total_count > 0:
        average_length = (
            sum(len(token) for token in tokens) / total_count
        )
        ttr = unique_count / total_count
    else:
        average_length = 0.0
        ttr = 0.0

    return {
        "Total number of tokens": total_count,
        "Number of unique tokens": unique_count,
        "Vocabulary size": unique_count,
        "Hapax legomena": hapax_count,
        "Number of rare tokens": rare_count,
        "Percentage of rare tokens": rare_percentage,
        "Average token length": average_length,
        "Type-token ratio": ttr
    }


def most_frequent_words(tokens, top_n):
    """Return the most frequent words."""
    return Counter(tokens).most_common(top_n)


def longest_tokens(tokens, top_n):
    """Return the longest unique tokens."""
    frequencies = Counter(tokens)

    ordered_tokens = sorted(
        frequencies.keys(),
        key=lambda token: (-len(token), token)
    )

    return [
        (token, len(token), frequencies[token])
        for token in ordered_tokens[:top_n]
    ]


def print_vocabulary_statistics(source_stats, target_stats):
    """Print the vocabulary statistics table."""
    report = pd.DataFrame({
        "Source text": source_stats,
        "Standardized text": target_stats
    })

    print("VOCABULARY STATISTICS")
    print("-" * 75)
    print(report.round(4).to_string())


def print_frequent_words(title, words):
    """Print the most frequent words."""
    print("\n" + title)
    print("-" * 50)

    for position, item in enumerate(words, start=1):
        word = item[0]
        frequency = item[1]

        print(f"{position:>2}. {word:<25} {frequency}")


def print_longest_tokens(title, tokens):
    """Print the longest tokens."""
    print("\n" + title)
    print("-" * 70)
    print(f"{'Token':<40} {'Length':>10} {'Frequency':>12}")
    print("-" * 70)

    for token, length, frequency in tokens:
        print(f"{token:<40} {length:>10} {frequency:>12}")


def main():
    df = load_dataset(FILE_PATH)

    source_tokens = get_all_tokens(df, "text")
    target_tokens = get_all_tokens(df, "standardized")

    source_stats = calculate_vocabulary_statistics(source_tokens)
    target_stats = calculate_vocabulary_statistics(target_tokens)

    print_vocabulary_statistics(source_stats, target_stats)

    print_frequent_words(
        "MOST FREQUENT SOURCE WORDS",
        most_frequent_words(source_tokens, TOP_N)
    )

    print_frequent_words(
        "MOST FREQUENT STANDARDIZED WORDS",
        most_frequent_words(target_tokens, TOP_N)
    )

    print_longest_tokens(
        "LONGEST SOURCE TOKENS",
        longest_tokens(source_tokens, TOP_N)
    )

    print_longest_tokens(
        "LONGEST STANDARDIZED TOKENS",
        longest_tokens(target_tokens, TOP_N)
    )


if __name__ == "__main__":
    main()

VOCABULARY STATISTICS
---------------------------------------------------------------------------
                           Source text  Standardized text
Total number of tokens     243171.0000        240655.0000
Number of unique tokens     16626.0000         11782.0000
Vocabulary size             16626.0000         11782.0000
Hapax legomena               4332.0000          2957.0000
Number of rare tokens        9869.0000          6817.0000
Percentage of rare tokens      59.3588            57.8594
Average token length            4.0718             4.0784
Type-token ratio                0.0684             0.0490

MOST FREQUENT SOURCE WORDS
--------------------------------------------------
 1. la                        9536
 2. sa                        6082
 3. li                        4802
 4. ki                        4347
 5. zot                       4308
 6. p                         3832
 7. pou                       3830
 8. name                      2541
 9. pa               

4. Source-to-target modification statistics

In [31]:
pip install rapidfuzz

   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 709.8 kB/s eta 0:00:02
   ------------------- -------------------- 0.8/1.6 MB 924.4 kB/s eta 0:00:01
   ------------------- -------------------- 0.8/1.6 MB 924.4 kB/s eta 0:00:01
   -------------------------- ------------- 1.0/1.6 MB 906.1 kB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 1.1 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import pandas as pd
from rapidfuzz.distance import Levenshtein


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def prepare_valid_pairs(df):
    """
    Keep records containing valid source and target strings.

    Missing values are excluded, but empty strings are retained.
    """
    required_columns = ["text", "standardized"]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    valid_df = df.dropna(
        subset=["text", "standardized"]
    ).copy()

    valid_df["text"] = valid_df["text"].astype(str)
    valid_df["standardized"] = (
        valid_df["standardized"].astype(str)
    )

    return valid_df


def calculate_character_edit_distance(source, target):
    """Calculate character-level Levenshtein edit distance."""
    return Levenshtein.distance(source, target)


def calculate_word_edit_distance(source, target):
    """
    Calculate word-level Levenshtein edit distance.

    Words are identified using whitespace separation.
    """
    source_tokens = source.split()
    target_tokens = target.split()

    return Levenshtein.distance(
        source_tokens,
        target_tokens
    )


def calculate_normalized_edit_distance(source, target):
    """
    Calculate normalized character-level edit distance.

    Distance = character edit distance divided by the
    maximum source or target character length.
    """
    maximum_length = max(len(source), len(target))

    if maximum_length == 0:
        return 0.0

    edit_distance = calculate_character_edit_distance(
        source,
        target
    )

    return edit_distance / maximum_length


def calculate_character_similarity(source, target):
    """
    Calculate character similarity from 0 to 1.

    1 means identical and 0 means completely dissimilar.
    """
    normalized_distance = (
        calculate_normalized_edit_distance(source, target)
    )

    return 1 - normalized_distance


def classify_modification(normalized_distance):
    """
    Classify the modification level using analytical thresholds.

    These are user-defined analytical thresholds and are not
    universal standards.
    """
    if normalized_distance == 0:
        return "Unchanged"

    if normalized_distance <= 0.20:
        return "Low"

    if normalized_distance <= 0.50:
        return "Medium"

    return "High"


def analyse_pairs(df):
    """Calculate source-to-target statistics for every valid pair."""
    analysis = prepare_valid_pairs(df)

    analysis["source_character_count"] = (
        analysis["text"].str.len()
    )

    analysis["target_character_count"] = (
        analysis["standardized"].str.len()
    )

    analysis["character_length_difference"] = (
        analysis["target_character_count"]
        - analysis["source_character_count"]
    )

    analysis["source_token_count"] = (
        analysis["text"].str.split().str.len()
    )

    analysis["target_token_count"] = (
        analysis["standardized"].str.split().str.len()
    )

    analysis["token_count_difference"] = (
        analysis["target_token_count"]
        - analysis["source_token_count"]
    )

    analysis["character_edit_distance"] = [
        calculate_character_edit_distance(source, target)
        for source, target in zip(
            analysis["text"],
            analysis["standardized"]
        )
    ]

    analysis["word_edit_distance"] = [
        calculate_word_edit_distance(source, target)
        for source, target in zip(
            analysis["text"],
            analysis["standardized"]
        )
    ]

    maximum_lengths = analysis[
        ["source_character_count", "target_character_count"]
    ].max(axis=1)

    analysis["normalized_edit_distance"] = (
        analysis["character_edit_distance"]
        .div(maximum_lengths)
        .fillna(0.0)
    )

    analysis["character_similarity_ratio"] = (
        1 - analysis["normalized_edit_distance"]
    )

    analysis["modification_level"] = (
        analysis["normalized_edit_distance"]
        .apply(classify_modification)
    )

    return analysis


def calculate_summary(analysis):
    """Calculate the overall modification statistics."""
    total_pairs = len(analysis)

    unchanged_count = (
        analysis["normalized_edit_distance"] == 0
    ).sum()

    modified_count = total_pairs - unchanged_count

    if total_pairs == 0:
        unchanged_percentage = 0.0
        modified_percentage = 0.0
    else:
        unchanged_percentage = (
            unchanged_count / total_pairs
        ) * 100

        modified_percentage = (
            modified_count / total_pairs
        ) * 100

    level_counts = (
        analysis["modification_level"]
        .value_counts()
    )

    low_count = level_counts.get("Low", 0)
    medium_count = level_counts.get("Medium", 0)
    high_count = level_counts.get("High", 0)

    if total_pairs == 0:
        low_percentage = 0.0
        medium_percentage = 0.0
        high_percentage = 0.0
    else:
        low_percentage = (low_count / total_pairs) * 100
        medium_percentage = (
            medium_count / total_pairs
        ) * 100
        high_percentage = (high_count / total_pairs) * 100

    summary = {
        "Total valid pairs": total_pairs,
        "Unchanged pairs": unchanged_count,
        "Unchanged percentage": unchanged_percentage,
        "Modified pairs": modified_count,
        "Modified percentage": modified_percentage,
        "Average character-length difference":
            analysis["character_length_difference"].mean(),
        "Average token-count difference":
            analysis["token_count_difference"].mean(),
        "Average character edit distance":
            analysis["character_edit_distance"].mean(),
        "Average word edit distance":
            analysis["word_edit_distance"].mean(),
        "Mean normalized edit distance":
            analysis["normalized_edit_distance"].mean(),
        "Median normalized edit distance":
            analysis["normalized_edit_distance"].median(),
        "Minimum normalized edit distance":
            analysis["normalized_edit_distance"].min(),
        "Maximum normalized edit distance":
            analysis["normalized_edit_distance"].max(),
        "Average character similarity ratio":
            analysis["character_similarity_ratio"].mean(),
        "Low modification pairs": low_count,
        "Low modification percentage": low_percentage,
        "Medium modification pairs": medium_count,
        "Medium modification percentage": medium_percentage,
        "High modification pairs": high_count,
        "High modification percentage": high_percentage
    }

    return summary


def print_summary(summary, original_count):
    """Print the source-to-target modification report."""
    valid_count = summary["Total valid pairs"]
    excluded_count = original_count - valid_count

    print("SOURCE-TO-TARGET MODIFICATION STATISTICS")
    print("-" * 65)

    print(f"Total records: {original_count}")
    print(f"Valid source-target pairs: {valid_count}")
    print(f"Pairs excluded because of missing values: {excluded_count}")

    print(
        f"\nUnchanged pairs: "
        f"{summary['Unchanged pairs']} "
        f"({summary['Unchanged percentage']:.2f}%)"
    )

    print(
        f"Modified pairs: "
        f"{summary['Modified pairs']} "
        f"({summary['Modified percentage']:.2f}%)"
    )

    print(
        f"\nAverage character-length difference: "
        f"{summary['Average character-length difference']:.2f}"
    )

    print(
        f"Average token-count difference: "
        f"{summary['Average token-count difference']:.2f}"
    )

    print(
        f"Average character edit distance: "
        f"{summary['Average character edit distance']:.2f}"
    )

    print(
        f"Average word edit distance: "
        f"{summary['Average word edit distance']:.2f}"
    )

    print(
        f"Mean normalized edit distance: "
        f"{summary['Mean normalized edit distance']:.4f}"
    )

    print(
        f"Median normalized edit distance: "
        f"{summary['Median normalized edit distance']:.4f}"
    )

    print(
        f"Minimum normalized edit distance: "
        f"{summary['Minimum normalized edit distance']:.4f}"
    )

    print(
        f"Maximum normalized edit distance: "
        f"{summary['Maximum normalized edit distance']:.4f}"
    )

    print(
        f"Average character similarity ratio: "
        f"{summary['Average character similarity ratio']:.4f} "
        f"({summary['Average character similarity ratio'] * 100:.2f}%)"
    )

    print("\nMODIFICATION LEVELS")
    print("-" * 65)

    print(
        f"Low modification (0 < distance <= 0.20): "
        f"{summary['Low modification pairs']} pairs "
        f"({summary['Low modification percentage']:.2f}%)"
    )

    print(
        f"Medium modification (0.20 < distance <= 0.50): "
        f"{summary['Medium modification pairs']} pairs "
        f"({summary['Medium modification percentage']:.2f}%)"
    )

    print(
        f"High modification (distance > 0.50): "
        f"{summary['High modification pairs']} pairs "
        f"({summary['High modification percentage']:.2f}%)"
    )


def main():
    df = load_dataset(FILE_PATH)

    analysis = analyse_pairs(df)
    summary = calculate_summary(analysis)

    print_summary(summary, len(df))


if __name__ == "__main__":
    main()

5. Token-level normalization analysis

In [32]:
import json
import re
import pandas as pd

from collections import Counter, defaultdict
from difflib import SequenceMatcher


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"

TOP_N = 30


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def tokenize(text):
    """
    Extract word tokens.

    Punctuation and emojis are excluded.
    Apostrophes inside words are preserved.
    """
    if pd.isna(text):
        return []

    return re.findall(
        r"\b\w+(?:['’]\w+)*\b",
        str(text),
        flags=re.UNICODE
    )


def analyse_sentence_pair(source_text, target_text):
    """Align the source and target tokens for one sentence pair."""

    source_tokens = tokenize(source_text)
    target_tokens = tokenize(target_text)

    # Use lowercase forms for comparison.
    # This prevents capitalization-only changes from being
    # counted as token normalization.
    source_comparison = [
        token.casefold() for token in source_tokens
    ]

    target_comparison = [
        token.casefold() for token in target_tokens
    ]

    matcher = SequenceMatcher(
        None,
        source_comparison,
        target_comparison,
        autojunk=False
    )

    mappings = []

    changed_source_tokens = 0
    deleted_source_tokens = 0
    inserted_target_tokens = 0

    for tag, source_start, source_end, target_start, target_end in (
        matcher.get_opcodes()
    ):
        # Identical tokens require no normalization.
        if tag == "equal":
            continue

        # Source tokens removed from the target.
        if tag == "delete":
            deleted_count = source_end - source_start

            changed_source_tokens += deleted_count
            deleted_source_tokens += deleted_count

            continue

        # Tokens added to the target.
        if tag == "insert":
            inserted_target_tokens += target_end - target_start

            continue

        # Replacement block.
        source_block = source_tokens[source_start:source_end]
        target_block = target_tokens[target_start:target_end]

        paired_count = min(
            len(source_block),
            len(target_block)
        )

        changed_source_tokens += len(source_block)

        deleted_source_tokens += max(
            0,
            len(source_block) - paired_count
        )

        inserted_target_tokens += max(
            0,
            len(target_block) - paired_count
        )

        # Create one-to-one normalization mappings.
        for position in range(paired_count):
            source_token = source_block[position].casefold()
            target_token = target_block[position].casefold()

            if source_token != target_token:
                mappings.append(
                    (source_token, target_token)
                )

    return {
        "source_token_count": len(source_tokens),
        "changed_source_tokens": changed_source_tokens,
        "deleted_source_tokens": deleted_source_tokens,
        "inserted_target_tokens": inserted_target_tokens,
        "mappings": mappings
    }


def analyse_dataset(df):
    """Analyse token changes across the complete dataset."""

    required_columns = ["text", "standardized"]

    for column in required_columns:
        if column not in df.columns:
            raise ValueError(
                f"Required column not found: {column}"
            )

    valid_df = df.dropna(
        subset=["text", "standardized"]
    )

    mapping_counts = Counter()

    total_source_tokens = 0
    total_changed_source_tokens = 0
    total_deleted_source_tokens = 0
    total_inserted_target_tokens = 0

    for source_text, target_text in zip(
        valid_df["text"],
        valid_df["standardized"]
    ):
        result = analyse_sentence_pair(
            source_text,
            target_text
        )

        total_source_tokens += (
            result["source_token_count"]
        )

        total_changed_source_tokens += (
            result["changed_source_tokens"]
        )

        total_deleted_source_tokens += (
            result["deleted_source_tokens"]
        )

        total_inserted_target_tokens += (
            result["inserted_target_tokens"]
        )

        mapping_counts.update(
            result["mappings"]
        )

    if total_source_tokens > 0:
        changed_percentage = (
            total_changed_source_tokens
            / total_source_tokens
        ) * 100
    else:
        changed_percentage = 0.0

    target_to_sources = defaultdict(set)
    source_to_targets = defaultdict(set)

    for source_token, target_token in mapping_counts:
        target_to_sources[target_token].add(source_token)
        source_to_targets[source_token].add(target_token)

    targets_with_multiple_sources = {
        target_token: sorted(source_tokens)
        for target_token, source_tokens
        in target_to_sources.items()
        if len(source_tokens) > 1
    }

    sources_with_multiple_targets = {
        source_token: sorted(target_tokens)
        for source_token, target_tokens
        in source_to_targets.items()
        if len(target_tokens) > 1
    }

    if target_to_sources:
        average_source_variants = (
            sum(
                len(source_tokens)
                for source_tokens
                in target_to_sources.values()
            )
            / len(target_to_sources)
        )
    else:
        average_source_variants = 0.0

    return {
        "total_source_tokens": total_source_tokens,
        "changed_source_tokens": (
            total_changed_source_tokens
        ),
        "changed_percentage": changed_percentage,
        "deleted_source_tokens": (
            total_deleted_source_tokens
        ),
        "inserted_target_tokens": (
            total_inserted_target_tokens
        ),
        "mapping_counts": mapping_counts,
        "unique_mappings": len(mapping_counts),
        "targets_with_multiple_sources": (
            targets_with_multiple_sources
        ),
        "sources_with_multiple_targets": (
            sources_with_multiple_targets
        ),
        "average_source_variants": (
            average_source_variants
        )
    }


def print_source_variants(
    grouped_mappings,
    mapping_counts,
    top_n
):
    """Print source variants mapping to the same target token."""

    print(
        "\nSOURCE VARIANTS MAPPING TO "
        "THE SAME STANDARDIZED TOKEN"
    )
    print("-" * 75)

    ranked_groups = sorted(
        grouped_mappings.items(),
        key=lambda item: (
            -len(item[1]),
            item[0]
        )
    )

    for target_token, source_tokens in ranked_groups[:top_n]:
        details = []

        for source_token in source_tokens:
            frequency = mapping_counts[
                (source_token, target_token)
            ]

            details.append(
                f"{source_token} ({frequency})"
            )

        print(
            f"{', '.join(details)} "
            f"-> {target_token}"
        )


def print_multiple_targets(
    grouped_mappings,
    mapping_counts,
    top_n
):
    """Print source tokens mapping to multiple target tokens."""

    print(
        "\nONE SOURCE FORM MAPPING TO "
        "MULTIPLE STANDARDIZED TOKENS"
    )
    print("-" * 75)

    ranked_groups = sorted(
        grouped_mappings.items(),
        key=lambda item: (
            -len(item[1]),
            item[0]
        )
    )

    for source_token, target_tokens in ranked_groups[:top_n]:
        details = []

        for target_token in target_tokens:
            frequency = mapping_counts[
                (source_token, target_token)
            ]

            details.append(
                f"{target_token} ({frequency})"
            )

        print(
            f"{source_token} -> "
            f"{', '.join(details)}"
        )


def print_results(results):
    """Print the token-level normalization results."""

    print("TOKEN-LEVEL NORMALIZATION ANALYSIS")
    print("-" * 75)

    print(
        f"Total source tokens: "
        f"{results['total_source_tokens']}"
    )

    print(
        f"Changed source tokens: "
        f"{results['changed_source_tokens']}"
    )

    print(
        f"Percentage of source tokens changed: "
        f"{results['changed_percentage']:.2f}%"
    )

    print(
        f"Number of unique normalization mappings: "
        f"{results['unique_mappings']}"
    )

    print(
        f"Deleted source tokens: "
        f"{results['deleted_source_tokens']}"
    )

    print(
        f"Inserted target tokens: "
        f"{results['inserted_target_tokens']}"
    )

    print(
        "Standardized tokens with multiple source variants: "
        f"{len(results['targets_with_multiple_sources'])}"
    )

    print(
        "Source forms with multiple standardized forms: "
        f"{len(results['sources_with_multiple_targets'])}"
    )

    print(
        "Average source variants per standardized token: "
        f"{results['average_source_variants']:.2f}"
    )

    print(
        f"\nTOP {TOP_N} NORMALIZATION MAPPINGS"
    )
    print("-" * 75)

    common_mappings = results[
        "mapping_counts"
    ].most_common(TOP_N)

    for rank, mapping_information in enumerate(
        common_mappings,
        start=1
    ):
        mapping = mapping_information[0]
        frequency = mapping_information[1]

        source_token = mapping[0]
        target_token = mapping[1]

        print(
            f"{rank:>2}. "
            f"{source_token:<20} -> "
            f"{target_token:<20} "
            f"{frequency}"
        )

    print_source_variants(
        results["targets_with_multiple_sources"],
        results["mapping_counts"],
        TOP_N
    )

    print_multiple_targets(
        results["sources_with_multiple_targets"],
        results["mapping_counts"],
        TOP_N
    )


def main():
    df = load_dataset(FILE_PATH)

    results = analyse_dataset(df)

    print_results(results)


if __name__ == "__main__":
    main()

TOKEN-LEVEL NORMALIZATION ANALYSIS
---------------------------------------------------------------------------
Total source tokens: 243171
Changed source tokens: 73790
Percentage of source tokens changed: 30.34%
Number of unique normalization mappings: 9076
Deleted source tokens: 3965
Inserted target tokens: 1449
Standardized tokens with multiple source variants: 1165
Source forms with multiple standardized forms: 818
Average source variants per standardized token: 3.23

TOP 30 NORMALIZATION MAPPINGS
---------------------------------------------------------------------------
 1. p                    -> pe                   3667
 2. pou                  -> pu                   3660
 3. ban                  -> bann                 1381
 4. ene                  -> enn                  1152
 5. tou                  -> tu                   830
 6. in                   -> inn                  826
 7. dimoune              -> dimunn               717
 8. ine                  -> inn            

6. Punctuation statistics

In [33]:
import json
import unicodedata
import pandas as pd

from collections import Counter
from difflib import SequenceMatcher


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"


CATEGORIES = [
    "Full stops",
    "Commas",
    "Question marks",
    "Exclamation marks",
    "Colons",
    "Semicolons",
    "Apostrophes",
    "Quotation marks",
    "Hyphens and dashes",
    "Ellipses",
    "Brackets and parentheses",
    "Other punctuation"
]


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def is_punctuation(character):
    """Check whether a character is Unicode punctuation."""

    return unicodedata.category(character).startswith("P")


def extract_punctuation(text):
    """
    Extract all Unicode punctuation from a text.

    Ellipsis forms such as .., ..., ...., … and ……
    are counted as one ellipsis occurrence.
    """

    if pd.isna(text):
        return []

    text = str(text)

    punctuation_tokens = []
    position = 0

    while position < len(text):
        character = text[position]

        # Capture consecutive ordinary full stops.
        if character == ".":
            end_position = position

            while (
                end_position < len(text)
                and text[end_position] == "."
            ):
                end_position += 1

            number_of_dots = end_position - position

            if number_of_dots >= 2:
                punctuation_tokens.append("ELLIPSIS")
            else:
                punctuation_tokens.append(".")

            position = end_position
            continue

        # Capture one or more Unicode ellipsis characters.
        if character == "…":
            end_position = position

            while (
                end_position < len(text)
                and text[end_position] == "…"
            ):
                end_position += 1

            punctuation_tokens.append("ELLIPSIS")

            position = end_position
            continue

        # Capture all other Unicode punctuation.
        if is_punctuation(character):
            punctuation_tokens.append(character)

        position += 1

    return punctuation_tokens


def punctuation_category(token):
    """Assign each punctuation token to a readable category."""

    if token == "ELLIPSIS":
        return "Ellipses"

    if token == ".":
        return "Full stops"

    if token == ",":
        return "Commas"

    if token == "?":
        return "Question marks"

    if token == "!":
        return "Exclamation marks"

    if token == ":":
        return "Colons"

    if token == ";":
        return "Semicolons"

    if token in {"'", "’"}:
        return "Apostrophes"

    unicode_category = unicodedata.category(token)
    unicode_name = unicodedata.name(token, "")

    if (
        unicode_category in {"Pi", "Pf"}
        or "QUOTATION MARK" in unicode_name
    ):
        return "Quotation marks"

    if (
        unicode_category == "Pd"
        or "HYPHEN" in unicode_name
        or "DASH" in unicode_name
    ):
        return "Hyphens and dashes"

    if unicode_category in {"Ps", "Pe"}:
        return "Brackets and parentheses"

    return "Other punctuation"


def count_by_category(tokens):
    """Count punctuation occurrences by category."""

    counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    for token in tokens:
        category = punctuation_category(token)
        counts[category] += 1

    return counts


def compare_punctuation(source_tokens, target_tokens):
    """
    Compare the source and target punctuation sequences.

    Calculate punctuation insertions, deletions and replacements.
    """

    matcher = SequenceMatcher(
        None,
        source_tokens,
        target_tokens,
        autojunk=False
    )

    added_tokens = Counter()
    removed_tokens = Counter()

    replaced_from = Counter()
    replaced_to = Counter()

    for (
        tag,
        source_start,
        source_end,
        target_start,
        target_end
    ) in matcher.get_opcodes():

        source_block = source_tokens[
            source_start:source_end
        ]

        target_block = target_tokens[
            target_start:target_end
        ]

        if tag == "equal":
            continue

        if tag == "insert":
            added_tokens.update(target_block)
            continue

        if tag == "delete":
            removed_tokens.update(source_block)
            continue

        if tag == "replace":
            paired_count = min(
                len(source_block),
                len(target_block)
            )

            # Punctuation tokens paired as replacements.
            replaced_from.update(
                source_block[:paired_count]
            )

            replaced_to.update(
                target_block[:paired_count]
            )

            # Additional source punctuation is removed.
            removed_tokens.update(
                source_block[paired_count:]
            )

            # Additional target punctuation is added.
            added_tokens.update(
                target_block[paired_count:]
            )

    added_total = sum(added_tokens.values())
    removed_total = sum(removed_tokens.values())
    replaced_total = sum(replaced_from.values())

    if (
        added_total == 0
        and removed_total == 0
        and replaced_total == 0
    ):
        classification = "No punctuation change"

    elif (
        added_total > 0
        and removed_total == 0
        and replaced_total == 0
    ):
        classification = "Punctuation insertion only"

    elif (
        removed_total > 0
        and added_total == 0
        and replaced_total == 0
    ):
        classification = "Punctuation deletion only"

    elif (
        replaced_total > 0
        and added_total == 0
        and removed_total == 0
    ):
        classification = "Punctuation replacement"

    else:
        classification = "Multiple punctuation changes"

    return {
        "added_tokens": added_tokens,
        "removed_tokens": removed_tokens,
        "replaced_from": replaced_from,
        "replaced_to": replaced_to,
        "classification": classification
    }


def get_final_punctuation(text):
    """
    Identify the final sentence punctuation.

    Recognised final marks:
    - Full stop
    - Question mark
    - Exclamation mark
    - Ellipsis

    Emojis, quotation marks and closing brackets are allowed
    after the sentence-final punctuation.

    Examples:
    twa?😊      returns ?
    li?😂😂😂   returns ?
    bon...      returns ELLIPSIS
    """

    if pd.isna(text):
        return None

    text = str(text).rstrip()

    punctuation_positions = []
    position = 0

    while position < len(text):
        character = text[position]

        if character == ".":
            end_position = position

            while (
                end_position < len(text)
                and text[end_position] == "."
            ):
                end_position += 1

            number_of_dots = end_position - position

            if number_of_dots >= 2:
                token = "ELLIPSIS"
            else:
                token = "."

            punctuation_positions.append(
                (token, end_position)
            )

            position = end_position
            continue

        if character == "…":
            end_position = position

            while (
                end_position < len(text)
                and text[end_position] == "…"
            ):
                end_position += 1

            punctuation_positions.append(
                ("ELLIPSIS", end_position)
            )

            position = end_position
            continue

        if character in {"?", "!"}:
            punctuation_positions.append(
                (character, position + 1)
            )

        position += 1

    # Check the possible final marks from right to left.
    for token, end_position in reversed(
        punctuation_positions
    ):
        trailing_text = text[end_position:]

        # If there is a letter or number after the punctuation,
        # the punctuation is not sentence-final.
        contains_text_after_mark = any(
            character.isalnum()
            for character in trailing_text
        )

        if not contains_text_after_mark:
            return token

    return None


def add_token_counts_to_categories(
    category_counter,
    token_counter
):
    """Convert punctuation-token counts into category counts."""

    for token, count in token_counter.items():
        category = punctuation_category(token)
        category_counter[category] += count


def analyse_punctuation(df):
    """Calculate punctuation statistics for the complete dataset."""

    required_columns = {
        "text",
        "standardized"
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    valid_df = df.dropna(
        subset=["text", "standardized"]
    )

    source_counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    target_counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    added_counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    removed_counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    replaced_from_counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    replaced_to_counts = Counter({
        category: 0
        for category in CATEGORIES
    })

    source_final_distribution = Counter()
    target_final_distribution = Counter()

    pair_classifications = Counter()

    source_sentences_with_punctuation = 0
    target_sentences_with_punctuation = 0

    final_punctuation_added = 0
    target_with_final_punctuation = 0

    for source_text, target_text in zip(
        valid_df["text"],
        valid_df["standardized"]
    ):
        source_tokens = extract_punctuation(
            source_text
        )

        target_tokens = extract_punctuation(
            target_text
        )

        if source_tokens:
            source_sentences_with_punctuation += 1

        if target_tokens:
            target_sentences_with_punctuation += 1

        source_counts.update(
            count_by_category(source_tokens)
        )

        target_counts.update(
            count_by_category(target_tokens)
        )

        comparison = compare_punctuation(
            source_tokens,
            target_tokens
        )

        pair_classifications[
            comparison["classification"]
        ] += 1

        add_token_counts_to_categories(
            added_counts,
            comparison["added_tokens"]
        )

        add_token_counts_to_categories(
            removed_counts,
            comparison["removed_tokens"]
        )

        add_token_counts_to_categories(
            replaced_from_counts,
            comparison["replaced_from"]
        )

        add_token_counts_to_categories(
            replaced_to_counts,
            comparison["replaced_to"]
        )

        source_final = get_final_punctuation(
            source_text
        )

        target_final = get_final_punctuation(
            target_text
        )

        source_final_distribution[
            source_final or "None"
        ] += 1

        target_final_distribution[
            target_final or "None"
        ] += 1

        if target_final is not None:
            target_with_final_punctuation += 1

        if (
            source_final is None
            and target_final is not None
        ):
            final_punctuation_added += 1

    total_pairs = len(valid_df)

    if total_pairs > 0:
        final_punctuation_percentage = (
            target_with_final_punctuation
            / total_pairs
        ) * 100
    else:
        final_punctuation_percentage = 0.0

    return {
        "valid_pairs": total_pairs,

        "source_sentences_with_punctuation":
            source_sentences_with_punctuation,

        "target_sentences_with_punctuation":
            target_sentences_with_punctuation,

        "source_counts": source_counts,
        "target_counts": target_counts,

        "added_counts": added_counts,
        "removed_counts": removed_counts,

        "replaced_from_counts":
            replaced_from_counts,

        "replaced_to_counts":
            replaced_to_counts,

        "source_final_distribution":
            source_final_distribution,

        "target_final_distribution":
            target_final_distribution,

        "pair_classifications":
            pair_classifications,

        "final_punctuation_added":
            final_punctuation_added,

        "target_with_final_punctuation":
            target_with_final_punctuation,

        "final_punctuation_percentage":
            final_punctuation_percentage
    }


def display_final_mark(mark):
    """Convert the internal ellipsis label into readable text."""

    if mark == "ELLIPSIS":
        return "Ellipsis"

    return mark


def print_results(results):
    """Print all punctuation statistics."""

    total_pairs = results["valid_pairs"]

    total_source_marks = sum(
        results["source_counts"].values()
    )

    total_target_marks = sum(
        results["target_counts"].values()
    )

    total_added = sum(
        results["added_counts"].values()
    )

    total_removed = sum(
        results["removed_counts"].values()
    )

    total_replaced = sum(
        results["replaced_from_counts"].values()
    )

    print("PUNCTUATION STATISTICS")
    print("-" * 90)

    print(
        f"Valid source-target pairs: "
        f"{total_pairs}"
    )

    print(
        f"Source sentences containing punctuation: "
        f"{results['source_sentences_with_punctuation']}"
    )

    print(
        f"Target sentences containing punctuation: "
        f"{results['target_sentences_with_punctuation']}"
    )

    print(
        f"Total source punctuation marks: "
        f"{total_source_marks}"
    )

    print(
        f"Total target punctuation marks: "
        f"{total_target_marks}"
    )

    print(
        f"Punctuation marks added: "
        f"{total_added}"
    )

    print(
        f"Punctuation marks removed: "
        f"{total_removed}"
    )

    print(
        f"Punctuation marks replaced: "
        f"{total_replaced}"
    )

    print(
        f"Sentence-final punctuation added: "
        f"{results['final_punctuation_added']}"
    )

    print(
        f"Target sentences with correct/restored "
        f"final punctuation: "
        f"{results['target_with_final_punctuation']} "
        f"({results['final_punctuation_percentage']:.2f}%)"
    )

    punctuation_table = pd.DataFrame({
        "Source": results["source_counts"],
        "Target": results["target_counts"],
        "Added": results["added_counts"],
        "Removed": results["removed_counts"],
        "Replaced from":
            results["replaced_from_counts"],
        "Replaced to":
            results["replaced_to_counts"]
    })

    punctuation_table = (
        punctuation_table
        .reindex(CATEGORIES)
        .fillna(0)
        .astype(int)
    )

    print("\nPUNCTUATION COUNTS BY CATEGORY")
    print("-" * 90)
    print(punctuation_table.to_string())

    print("\nSOURCE FINAL-PUNCTUATION DISTRIBUTION")
    print("-" * 90)

    for mark, count in (
        results["source_final_distribution"]
        .most_common()
    ):
        if total_pairs > 0:
            percentage = (
                count / total_pairs
            ) * 100
        else:
            percentage = 0.0

        print(
            f"{display_final_mark(mark):<15} "
            f"{count:>8} "
            f"({percentage:.2f}%)"
        )

    print("\nTARGET FINAL-PUNCTUATION DISTRIBUTION")
    print("-" * 90)

    for mark, count in (
        results["target_final_distribution"]
        .most_common()
    ):
        if total_pairs > 0:
            percentage = (
                count / total_pairs
            ) * 100
        else:
            percentage = 0.0

        print(
            f"{display_final_mark(mark):<15} "
            f"{count:>8} "
            f"({percentage:.2f}%)"
        )

    print("\nPAIR CLASSIFICATION")
    print("-" * 90)

    classification_order = [
        "No punctuation change",
        "Punctuation insertion only",
        "Punctuation deletion only",
        "Punctuation replacement",
        "Multiple punctuation changes"
    ]

    for classification in classification_order:
        count = results[
            "pair_classifications"
        ].get(classification, 0)

        if total_pairs > 0:
            percentage = (
                count / total_pairs
            ) * 100
        else:
            percentage = 0.0

        print(
            f"{classification:<35} "
            f"{count:>8} "
            f"({percentage:.2f}%)"
        )


def main():
    df = load_dataset(FILE_PATH)

    results = analyse_punctuation(df)

    print_results(results)


if __name__ == "__main__":
    main()

PUNCTUATION STATISTICS
------------------------------------------------------------------------------------------
Valid source-target pairs: 13830
Source sentences containing punctuation: 7106
Target sentences containing punctuation: 13830
Total source punctuation marks: 21874
Total target punctuation marks: 47410
Punctuation marks added: 26330
Punctuation marks removed: 794
Punctuation marks replaced: 773
Sentence-final punctuation added: 11850
Target sentences with correct/restored final punctuation: 13716 (99.18%)

PUNCTUATION COUNTS BY CATEGORY
------------------------------------------------------------------------------------------
                          Source  Target  Added  Removed  Replaced from  Replaced to
Full stops                  3857   20539  16654      349             71          448
Commas                      4363    9638   5626      225            252          126
Question marks              1984    4397   2330       13              4          100
Exclamation ma

7. Capitalization statistics

In [34]:
import json
import unicodedata
import pandas as pd

from collections import Counter
from difflib import SequenceMatcher


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def normalize_text(text):
    """Apply Unicode normalization to a text."""

    if pd.isna(text):
        return ""

    return unicodedata.normalize(
        "NFC",
        str(text)
    )


def get_first_alphabetic_character(text):
    """
    Return the first alphabetic character in a text.

    Spaces, punctuation, numbers, brackets and emojis
    appearing before the first letter are ignored.
    """

    text = normalize_text(text)

    for character in text:
        if character.isalpha():
            return character

    return None


def begins_with_lowercase_letter(text):
    """
    Check whether the first alphabetic character
    is lowercase.
    """

    first_letter = get_first_alphabetic_character(
        text
    )

    if first_letter is None:
        return False

    return first_letter.islower()


def begins_with_uppercase_letter(text):
    """
    Check whether the first alphabetic character
    is uppercase.
    """

    first_letter = get_first_alphabetic_character(
        text
    )

    if first_letter is None:
        return False

    return first_letter.isupper()


def is_initial_capitalization_correction(
    source_text,
    target_text
):
    """
    Check whether sentence-initial capitalization
    was corrected.

    The first source letter must be lowercase.
    The first target letter must be uppercase.
    Both letters must otherwise be the same.
    """

    source_letter = get_first_alphabetic_character(
        source_text
    )

    target_letter = get_first_alphabetic_character(
        target_text
    )

    if source_letter is None or target_letter is None:
        return False

    same_letter = (
        source_letter.casefold()
        == target_letter.casefold()
    )

    return (
        same_letter
        and source_letter.islower()
        and target_letter.isupper()
    )


def classify_sentence_case(text):
    """
    Classify text as fully uppercase, fully lowercase,
    mixed case or containing no cased letters.

    Only alphabetic letters that have uppercase/lowercase
    forms are considered.
    """

    text = normalize_text(text)

    cased_letters = [
        character
        for character in text
        if (
            character.isalpha()
            and (
                character.islower()
                or character.isupper()
            )
        )
    ]

    if not cased_letters:
        return "No cased letters"

    if all(
        character.isupper()
        for character in cased_letters
    ):
        return "Fully uppercase"

    if all(
        character.islower()
        for character in cased_letters
    ):
        return "Fully lowercase"

    return "Mixed case"


def count_character_case_changes(
    source_text,
    target_text
):
    """
    Count lowercase-to-uppercase and
    uppercase-to-lowercase character changes.

    Characters are aligned without considering case.
    Changes caused by spelling differences are not
    counted as capitalization changes.
    """

    source_text = normalize_text(source_text)
    target_text = normalize_text(target_text)

    source_comparison = [
        character.casefold()
        for character in source_text
    ]

    target_comparison = [
        character.casefold()
        for character in target_text
    ]

    matcher = SequenceMatcher(
        None,
        source_comparison,
        target_comparison,
        autojunk=False
    )

    lowercase_to_uppercase = 0
    uppercase_to_lowercase = 0

    for (
        tag,
        source_start,
        source_end,
        target_start,
        target_end
    ) in matcher.get_opcodes():

        # Only equal case-insensitive blocks represent
        # the same underlying characters.
        if tag != "equal":
            continue

        source_block = source_text[
            source_start:source_end
        ]

        target_block = target_text[
            target_start:target_end
        ]

        for source_character, target_character in zip(
            source_block,
            target_block
        ):
            if (
                source_character.isalpha()
                and target_character.isalpha()
            ):
                if (
                    source_character.islower()
                    and target_character.isupper()
                ):
                    lowercase_to_uppercase += 1

                elif (
                    source_character.isupper()
                    and target_character.islower()
                ):
                    uppercase_to_lowercase += 1

    return (
        lowercase_to_uppercase,
        uppercase_to_lowercase
    )


def is_capitalization_only_transformation(
    source_text,
    target_text
):
    """
    Check whether source and target differ only by letter case.

    Punctuation, spelling, spacing and emojis must otherwise
    remain exactly the same.
    """

    source_text = normalize_text(source_text)
    target_text = normalize_text(target_text)

    if source_text == target_text:
        return False

    return (
        source_text.casefold()
        == target_text.casefold()
    )


def analyse_capitalization(df):
    """Calculate capitalization statistics."""

    required_columns = {
        "text",
        "standardized"
    }

    missing_columns = required_columns.difference(
        df.columns
    )

    if missing_columns:
        raise ValueError(
            f"Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    valid_df = df.dropna(
        subset=["text", "standardized"]
    )

    source_case_distribution = Counter()
    target_case_distribution = Counter()

    source_beginning_lowercase = 0
    target_beginning_uppercase = 0

    initial_capitalization_corrections = 0

    lowercase_to_uppercase_changes = 0
    uppercase_to_lowercase_changes = 0

    capitalization_only_transformations = 0

    source_without_alphabetic_characters = 0
    target_without_alphabetic_characters = 0

    for source_text, target_text in zip(
        valid_df["text"],
        valid_df["standardized"]
    ):
        source_first_letter = (
            get_first_alphabetic_character(
                source_text
            )
        )

        target_first_letter = (
            get_first_alphabetic_character(
                target_text
            )
        )

        if source_first_letter is None:
            source_without_alphabetic_characters += 1

        if target_first_letter is None:
            target_without_alphabetic_characters += 1

        if begins_with_lowercase_letter(source_text):
            source_beginning_lowercase += 1

        if begins_with_uppercase_letter(target_text):
            target_beginning_uppercase += 1

        if is_initial_capitalization_correction(
            source_text,
            target_text
        ):
            initial_capitalization_corrections += 1

        (
            lower_to_upper,
            upper_to_lower
        ) = count_character_case_changes(
            source_text,
            target_text
        )

        lowercase_to_uppercase_changes += (
            lower_to_upper
        )

        uppercase_to_lowercase_changes += (
            upper_to_lower
        )

        source_case = classify_sentence_case(
            source_text
        )

        target_case = classify_sentence_case(
            target_text
        )

        source_case_distribution[source_case] += 1
        target_case_distribution[target_case] += 1

        if is_capitalization_only_transformation(
            source_text,
            target_text
        ):
            capitalization_only_transformations += 1

    total_pairs = len(valid_df)

    return {
        "total_pairs": total_pairs,

        "source_beginning_lowercase":
            source_beginning_lowercase,

        "target_beginning_uppercase":
            target_beginning_uppercase,

        "initial_capitalization_corrections":
            initial_capitalization_corrections,

        "lowercase_to_uppercase_changes":
            lowercase_to_uppercase_changes,

        "uppercase_to_lowercase_changes":
            uppercase_to_lowercase_changes,

        "capitalization_only_transformations":
            capitalization_only_transformations,

        "source_case_distribution":
            source_case_distribution,

        "target_case_distribution":
            target_case_distribution,

        "source_without_alphabetic_characters":
            source_without_alphabetic_characters,

        "target_without_alphabetic_characters":
            target_without_alphabetic_characters
    }


def percentage(count, total):
    """Calculate a percentage safely."""

    if total == 0:
        return 0.0

    return (count / total) * 100


def print_results(results):
    """Print the capitalization statistics."""

    total_pairs = results["total_pairs"]

    print("CAPITALIZATION STATISTICS")
    print("-" * 85)

    print(
        f"Valid source-target pairs: "
        f"{total_pairs}"
    )

    print(
        f"Source sentences beginning with lowercase: "
        f"{results['source_beginning_lowercase']} "
        f"({percentage(
            results['source_beginning_lowercase'],
            total_pairs
        ):.2f}%)"
    )

    print(
        f"Target sentences beginning with uppercase: "
        f"{results['target_beginning_uppercase']} "
        f"({percentage(
            results['target_beginning_uppercase'],
            total_pairs
        ):.2f}%)"
    )

    print(
        f"Initial capitalization corrections: "
        f"{results['initial_capitalization_corrections']} "
        f"({percentage(
            results['initial_capitalization_corrections'],
            total_pairs
        ):.2f}%)"
    )

    print(
        f"Lowercase-to-uppercase character changes: "
        f"{results['lowercase_to_uppercase_changes']}"
    )

    print(
        f"Uppercase-to-lowercase character changes: "
        f"{results['uppercase_to_lowercase_changes']}"
    )

    print(
        f"Capitalization-only transformations: "
        f"{results['capitalization_only_transformations']} "
        f"({percentage(
            results['capitalization_only_transformations'],
            total_pairs
        ):.2f}%)"
    )

    print(
        f"Source texts without alphabetic characters: "
        f"{results['source_without_alphabetic_characters']}"
    )

    print(
        f"Target texts without alphabetic characters: "
        f"{results['target_without_alphabetic_characters']}"
    )

    case_order = [
        "Fully uppercase",
        "Fully lowercase",
        "Mixed case",
        "No cased letters"
    ]

    case_table = pd.DataFrame({
        "Source sentences": {
            category:
                results[
                    "source_case_distribution"
                ].get(category, 0)
            for category in case_order
        },

        "Target sentences": {
            category:
                results[
                    "target_case_distribution"
                ].get(category, 0)
            for category in case_order
        }
    })

    case_table["Source percentage"] = (
        case_table["Source sentences"]
        / total_pairs
        * 100
        if total_pairs > 0
        else 0.0
    )

    case_table["Target percentage"] = (
        case_table["Target sentences"]
        / total_pairs
        * 100
        if total_pairs > 0
        else 0.0
    )

    print("\nSENTENCE CASE DISTRIBUTION")
    print("-" * 85)
    print(case_table.round(2).to_string())


def main():
    df = load_dataset(FILE_PATH)

    results = analyse_capitalization(df)

    print_results(results)


if __name__ == "__main__":
    main()

CAPITALIZATION STATISTICS
-------------------------------------------------------------------------------------
Valid source-target pairs: 13830
Source sentences beginning with lowercase: 536 (3.88%)
Target sentences beginning with uppercase: 13566 (98.09%)
Initial capitalization corrections: 278 (2.01%)
Lowercase-to-uppercase character changes: 13267
Uppercase-to-lowercase character changes: 2324
Capitalization-only transformations: 20 (0.14%)
Source texts without alphabetic characters: 1
Target texts without alphabetic characters: 1

SENTENCE CASE DISTRIBUTION
-------------------------------------------------------------------------------------
                  Source sentences  Target sentences  Source percentage  Target percentage
Fully uppercase                167               154               1.21               1.11
Fully lowercase                407                93               2.94               0.67
Mixed case                   13255             13582              95.84 

8. Emoji and symbol analysis

In [35]:
%pip install emoji

   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/608.4 kB ? eta -:--:--
   ---------------------------------------- 608.4/608.4 kB 756.2 kB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import json
import unicodedata
import pandas as pd
import emoji

from collections import Counter


FILE_PATH = r"C:\Users\VE296WM\Downloads\diss\mc_joint_text_restoration.json"

TOP_N = 20


def load_dataset(file_path):
    """Load the JSON dataset into a DataFrame."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return pd.DataFrame(data)


def extract_emojis(text):
    """Extract complete emoji sequences from a text."""
    if pd.isna(text):
        return []

    return [
        item["emoji"]
        for item in emoji.emoji_list(str(text))
    ]


def extract_emojis_with_positions(text):
    """
    Extract emojis and calculate their relative positions.

    Relative position:
    0 = beginning of text
    1 = end of text
    """
    if pd.isna(text):
        return []

    text = str(text)

    if len(text) == 0:
        return []

    results = []

    for item in emoji.emoji_list(text):
        results.append({
            "emoji": item["emoji"],
            "relative_position": item["match_start"] / len(text)
        })

    return results


def contains_emoji(text):
    """Return True if the text contains at least one emoji."""
    return len(extract_emojis(text)) > 0


def get_emoji_frequencies(text_series):
    """Count every emoji occurrence in a text column."""
    frequencies = Counter()

    for text in text_series:
        frequencies.update(extract_emojis(text))

    return frequencies


def compare_emojis(source_text, target_text):
    """
    Compare source and target emojis.

    Repeated emojis are counted separately.

    Example:
    Source: 😂😂😊
    Target: 😂😊

    Preserved: 2
    Removed: 1
    Added: 0
    """
    source_emojis = extract_emojis(source_text)
    target_emojis = extract_emojis(target_text)

    source_counts = Counter(source_emojis)
    target_counts = Counter(target_emojis)

    preserved = source_counts & target_counts
    removed = source_counts - target_counts
    added = target_counts - source_counts

    return {
        "source_emojis": source_emojis,
        "target_emojis": target_emojis,
        "preserved": preserved,
        "removed": removed,
        "added": added,
        "preserved_total": sum(preserved.values()),
        "removed_total": sum(removed.values()),
        "added_total": sum(added.values())
    }


def emoji_position_changed(source_text, target_text, tolerance=0.05):
    """
    Check whether preserved emojis changed relative position.

    The emoji sequences must be identical.
    A difference greater than 5% of text length is counted
    as a position change.
    """
    source_positions = extract_emojis_with_positions(source_text)
    target_positions = extract_emojis_with_positions(target_text)

    source_sequence = [
        item["emoji"]
        for item in source_positions
    ]

    target_sequence = [
        item["emoji"]
        for item in target_positions
    ]

    if not source_sequence:
        return False

    # Added, removed or reordered emojis are handled separately.
    if source_sequence != target_sequence:
        return False

    for source_item, target_item in zip(
        source_positions,
        target_positions
    ):
        difference = abs(
            source_item["relative_position"]
            - target_item["relative_position"]
        )

        if difference > tolerance:
            return True

    return False


def emoji_order_changed(source_text, target_text):
    """
    Check whether the same emojis appear in a different order.
    """
    source_emojis = extract_emojis(source_text)
    target_emojis = extract_emojis(target_text)

    if not source_emojis:
        return False

    same_counts = (
        Counter(source_emojis)
        == Counter(target_emojis)
    )

    different_order = (
        source_emojis != target_emojis
    )

    return same_counts and different_order


def extract_other_symbols(text):
    """
    Extract non-alphanumeric Unicode symbols excluding emojis.

    Examples may include:
    €, £, $, +, =, ©, ™
    """
    if pd.isna(text):
        return []

    # Remove complete emoji sequences first.
    text_without_emojis = emoji.replace_emoji(
        str(text),
        replace=""
    )

    symbols = []

    for character in text_without_emojis:
        unicode_category = unicodedata.category(character)

        # Unicode symbol categories begin with S.
        if unicode_category.startswith("S"):
            symbols.append(character)

    return symbols


def get_symbol_frequencies(text_series):
    """Count other non-alphanumeric symbols."""
    frequencies = Counter()

    for text in text_series:
        frequencies.update(extract_other_symbols(text))

    return frequencies


def analyse_emojis_and_symbols(df):
    """Calculate emoji and symbol statistics."""

    required_columns = {"text", "standardized"}
    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    valid_df = df.dropna(
        subset=["text", "standardized"]
    )

    total_pairs = len(valid_df)

    source_frequencies = get_emoji_frequencies(
        valid_df["text"]
    )

    target_frequencies = get_emoji_frequencies(
        valid_df["standardized"]
    )

    source_symbol_frequencies = get_symbol_frequencies(
        valid_df["text"]
    )

    target_symbol_frequencies = get_symbol_frequencies(
        valid_df["standardized"]
    )

    total_source_emojis = sum(source_frequencies.values())
    total_target_emojis = sum(target_frequencies.values())

    source_sentences_with_emojis = sum(
        contains_emoji(text)
        for text in valid_df["text"]
    )

    target_sentences_with_emojis = sum(
        contains_emoji(text)
        for text in valid_df["standardized"]
    )

    preserved_frequencies = Counter()
    added_frequencies = Counter()
    removed_frequencies = Counter()

    preserved_total = 0
    added_total = 0
    removed_total = 0

    pairs_with_emoji_addition = 0
    pairs_with_emoji_removal = 0
    pairs_with_all_emojis_preserved = 0
    pairs_with_position_change = 0
    pairs_with_order_change = 0

    for source_text, target_text in zip(
        valid_df["text"],
        valid_df["standardized"]
    ):
        comparison = compare_emojis(
            source_text,
            target_text
        )

        preserved_total += comparison["preserved_total"]
        added_total += comparison["added_total"]
        removed_total += comparison["removed_total"]

        preserved_frequencies.update(
            comparison["preserved"]
        )

        added_frequencies.update(
            comparison["added"]
        )

        removed_frequencies.update(
            comparison["removed"]
        )

        source_emojis = comparison["source_emojis"]
        target_emojis = comparison["target_emojis"]

        if (
            source_emojis
            and Counter(source_emojis)
            == Counter(target_emojis)
        ):
            pairs_with_all_emojis_preserved += 1

        if comparison["added_total"] > 0:
            pairs_with_emoji_addition += 1

        if comparison["removed_total"] > 0:
            pairs_with_emoji_removal += 1

        if emoji_position_changed(
            source_text,
            target_text
        ):
            pairs_with_position_change += 1

        if emoji_order_changed(
            source_text,
            target_text
        ):
            pairs_with_order_change += 1

    if total_pairs > 0:
        source_sentence_percentage = (
            source_sentences_with_emojis
            / total_pairs
        ) * 100

        target_sentence_percentage = (
            target_sentences_with_emojis
            / total_pairs
        ) * 100
    else:
        source_sentence_percentage = 0.0
        target_sentence_percentage = 0.0

    if total_source_emojis > 0:
        preservation_percentage = (
            preserved_total
            / total_source_emojis
        ) * 100
    else:
        preservation_percentage = 0.0

    return {
        "total_pairs": total_pairs,
        "total_source_emojis": total_source_emojis,
        "total_target_emojis": total_target_emojis,
        "source_sentences_with_emojis":
            source_sentences_with_emojis,
        "target_sentences_with_emojis":
            target_sentences_with_emojis,
        "source_sentence_percentage":
            source_sentence_percentage,
        "target_sentence_percentage":
            target_sentence_percentage,
        "source_frequencies": source_frequencies,
        "target_frequencies": target_frequencies,
        "preserved_total": preserved_total,
        "preservation_percentage":
            preservation_percentage,
        "added_total": added_total,
        "removed_total": removed_total,
        "preserved_frequencies": preserved_frequencies,
        "added_frequencies": added_frequencies,
        "removed_frequencies": removed_frequencies,
        "pairs_with_all_emojis_preserved":
            pairs_with_all_emojis_preserved,
        "pairs_with_emoji_addition":
            pairs_with_emoji_addition,
        "pairs_with_emoji_removal":
            pairs_with_emoji_removal,
        "pairs_with_position_change":
            pairs_with_position_change,
        "pairs_with_order_change":
            pairs_with_order_change,
        "source_symbol_frequencies":
            source_symbol_frequencies,
        "target_symbol_frequencies":
            target_symbol_frequencies
    }


def calculate_percentage(count, total):
    """Calculate a percentage safely."""
    if total == 0:
        return 0.0

    return (count / total) * 100


def print_frequencies(title, frequencies):
    """Print the most frequent emojis or symbols."""
    print("\n" + title)
    print("-" * 60)

    if not frequencies:
        print("None found")
        return

    for position, item in enumerate(
        frequencies.most_common(TOP_N),
        start=1
    ):
        symbol = item[0]
        frequency = item[1]

        print(
            f"{position:>2}. "
            f"{symbol:<10} "
            f"{frequency}"
        )


def print_results(results):
    """Print the complete emoji and symbol report."""

    total_pairs = results["total_pairs"]

    print("EMOJI AND SYMBOL ANALYSIS")
    print("-" * 75)

    print(
        f"Valid source-target pairs: "
        f"{total_pairs}"
    )

    print(
        f"Total emojis in source: "
        f"{results['total_source_emojis']}"
    )

    print(
        f"Total emojis in target: "
        f"{results['total_target_emojis']}"
    )

    print(
        f"Source sentences containing emojis: "
        f"{results['source_sentences_with_emojis']} "
        f"({results['source_sentence_percentage']:.2f}%)"
    )

    print(
        f"Target sentences containing emojis: "
        f"{results['target_sentences_with_emojis']} "
        f"({results['target_sentence_percentage']:.2f}%)"
    )

    print(
        f"Emojis preserved during standardization: "
        f"{results['preserved_total']} "
        f"({results['preservation_percentage']:.2f}% "
        f"of source emojis)"
    )

    print(
        f"Emojis added: "
        f"{results['added_total']}"
    )

    print(
        f"Emojis removed: "
        f"{results['removed_total']}"
    )

    preserved_pairs = (
        results["pairs_with_all_emojis_preserved"]
    )

    print(
        f"Pairs where all source emojis were preserved: "
        f"{preserved_pairs} "
        f"({calculate_percentage(
            preserved_pairs,
            total_pairs
        ):.2f}%)"
    )

    addition_pairs = (
        results["pairs_with_emoji_addition"]
    )

    print(
        f"Pairs containing emoji additions: "
        f"{addition_pairs} "
        f"({calculate_percentage(
            addition_pairs,
            total_pairs
        ):.2f}%)"
    )

    removal_pairs = (
        results["pairs_with_emoji_removal"]
    )

    print(
        f"Pairs containing emoji removals: "
        f"{removal_pairs} "
        f"({calculate_percentage(
            removal_pairs,
            total_pairs
        ):.2f}%)"
    )

    position_pairs = (
        results["pairs_with_position_change"]
    )

    print(
        f"Pairs where emoji position changed: "
        f"{position_pairs} "
        f"({calculate_percentage(
            position_pairs,
            total_pairs
        ):.2f}%)"
    )

    order_pairs = (
        results["pairs_with_order_change"]
    )

    print(
        f"Pairs where emoji order changed: "
        f"{order_pairs} "
        f"({calculate_percentage(
            order_pairs,
            total_pairs
        ):.2f}%)"
    )

    print_frequencies(
        "MOST FREQUENT SOURCE EMOJIS",
        results["source_frequencies"]
    )

    print_frequencies(
        "MOST FREQUENT TARGET EMOJIS",
        results["target_frequencies"]
    )

    print_frequencies(
        "MOST FREQUENT PRESERVED EMOJIS",
        results["preserved_frequencies"]
    )

    print_frequencies(
        "EMOJIS ADDED DURING STANDARDIZATION",
        results["added_frequencies"]
    )

    print_frequencies(
        "EMOJIS REMOVED DURING STANDARDIZATION",
        results["removed_frequencies"]
    )

    print_frequencies(
        "OTHER NON-ALPHANUMERIC SYMBOLS IN SOURCE",
        results["source_symbol_frequencies"]
    )

    print_frequencies(
        "OTHER NON-ALPHANUMERIC SYMBOLS IN TARGET",
        results["target_symbol_frequencies"]
    )


def main():
    df = load_dataset(FILE_PATH)

    results = analyse_emojis_and_symbols(df)

    print_results(results)


if __name__ == "__main__":
    main()

EMOJI AND SYMBOL ANALYSIS
---------------------------------------------------------------------------
Valid source-target pairs: 13830
Total emojis in source: 6332
Total emojis in target: 6325
Source sentences containing emojis: 2585 (18.69%)
Target sentences containing emojis: 2583 (18.68%)
Emojis preserved during standardization: 6325 (99.89% of source emojis)
Emojis added: 0
Emojis removed: 7
Pairs where all source emojis were preserved: 2578 (18.64%)
Pairs containing emoji additions: 0 (0.00%)
Pairs containing emoji removals: 7 (0.05%)
Pairs where emoji position changed: 23 (0.17%)
Pairs where emoji order changed: 0 (0.00%)

MOST FREQUENT SOURCE EMOJIS
------------------------------------------------------------
 1. 🤣          1758
 2. 😂          999
 3. 🤔          263
 4. 🙏          240
 5. 🙄          230
 6. 😡          201
 7. 😅          195
 8. 😭          105
 9. 😆          79
10. 😢          78
11. 😁          77
12. 😒          60
13. 🤬          56
14. 💯          49
15. 😈        